# Cleaning and exploratory analysis

## 1. Setup

In [ ]:
#packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
# Run from the project root or from the 02 Scripts folder.
folder_path = Path.cwd()
if folder_path.name == "02 Scripts":
    folder_path = folder_path.parent

input_path = folder_path / "01 Input"
temp_path = folder_path / "03 Temp"
table_path = folder_path / "04 Tables"
chart_path = folder_path / "05 Charts"

if not input_path.is_dir():
    raise FileNotFoundError(f"Input folder not found: {input_path}")
for output_path in (temp_path, table_path, chart_path):
    output_path.mkdir(exist_ok=True)

In [ ]:
# chart colour

chart_colour = "#1F3A5F"

In [ ]:
# load data

training_df = pd.read_csv(input_path / "Train.csv")
test_df = pd.read_csv(input_path / "Test.csv")

## 2. Initial checks

In [ ]:
# quick overview of the data

print("Training head:")
display(training_df.head())

print("Test head:")
display(test_df.head())

print("Training shape:", training_df.shape)
print("Test shape:", test_df.shape)

print("Training variable types:")
display(training_df.dtypes)

print("Training unique values per variable")
display(training_df.nunique(dropna=True))

## 3. Training data cleaning

In [ ]:
# Drop apparent identifier columns
# "Unnamed: 0" looks like an exported row index, and "id" is unique per observation
id_variables = ["Unnamed: 0", "id"]

training_df = training_df.drop(columns = id_variables)
test_df = test_df.drop(columns = id_variables)

In [ ]:
#check the outcome poverty variable is boolean in both datasets (important for modelling later)
print(training_df["poor"].dtype)
print(training_df["poor"].value_counts(dropna=False))

print(test_df["poor"].dtype)
print(test_df["poor"].value_counts(dropna=False))

In [ ]:
#Define numeric_cols, categoric_cols
categorical_cols = ["kjkrfgld", "bpowgknt", "raksnhjf", "vwpsxrgk"]
numeric_cols = ["omtioxzz", "yfmzwkru", "tiwrsloh", "weioazcf"]

In [ ]:
# check no unseen categories in the test data
# if there are, this should not break the code because OneHotEncoder(handle_unknown="ignore") handles unseen categories
for col in categorical_cols:
    train_levels = set(training_df[col].dropna().unique())
    test_levels = set(test_df[col].dropna().unique())

    print(col)
    print("in test not train:", sorted(test_levels - train_levels))
    print("in train not test:", sorted(train_levels - test_levels))
    

In [ ]:
# given numeric variables are binned, check values taken by each numeric variable to see if any strange/impossible values

for col in numeric_cols:
    print(f"\n{col}")

    counts = (
        training_df[col]
        .value_counts(dropna=False)
        .sort_index()
        .reset_index(name="count")
        .rename(columns={col: "value"})
    )

    display(counts)
    

In [ ]:
#replace the clearly impossible values as missing in both datasets
# (this still avoids leakage as, even though i replace values in the test data, I have not learnt the impossible values from the test data, only the training data)

invalid_values = {
    "omtioxzz": [-127],
    "yfmzwkru": [1000]
}

for col, values in invalid_values.items():
    training_df[col] = training_df[col].replace(values, np.nan)
    test_df[col] = test_df[col].replace(values, np.nan)

# Check replacement successful
for col, values in invalid_values.items():
    assert not training_df[col].isin(values).any()
    assert not test_df[col].isin(values).any()

## 4. Exploratory charts and correlation checks

In [ ]:
# Plot chart showing poverty rates in the training data

target_balance = training_df["poor"].value_counts(normalize=True).sort_index()
target_balance.index = target_balance.index.astype(str)
target_balance = target_balance * 100

display(target_balance.reset_index(name="share").rename(columns={"poor": "poor"}))

ax = target_balance.plot(kind="bar", color=chart_colour)

ax.set_ylabel("Share (%)")
ax.set_xlabel("Poor")

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(chart_path / "target_balance_training.png")
plt.show()

In [ ]:
# examine correlation between numeric variables with a plot

corr = training_df[numeric_cols].corr()

display(corr)

plt.figure()
sns.heatmap(corr, annot=True, vmin=-1, vmax=1)
plt.tight_layout()
plt.savefig(chart_path / "numeric_correlation_heatmap.png")
plt.show()

In [ ]:
# Confirm near 1 correlation is not rounding. Drop if appear to be perfectly collinear

corr = training_df["tiwrsloh"].corr(training_df["weioazcf"])

print(f"Correlation: {corr:.12f}")

# Given correlation of nearly 1, appears they are linear transformations of each other
# Therefore dropone of the linear pair of variables - i choose weioazcf

training_df = training_df.drop(columns="weioazcf")
test_df = test_df.drop(columns="weioazcf")

numeric_cols = [col for col in numeric_cols if col != "weioazcf"]
feature_cols = numeric_cols + categorical_cols

# Confirm drop
print("Numeric columns:", numeric_cols)
print("Training shape:", training_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
# numeric variable distributions - training data only

# as well as giving a sense of the distribution of the data (including outliers)
# this serves as a useful check of whether the numeric variables are really numeric rather than encoded categorical
# variables (which i don't know given the data is anonymised)

for col in numeric_cols:
    x = training_df[col].dropna()
    bins = min(20, x.nunique())

    plt.figure()
    plt.hist(
        x,
        bins=bins,
        weights=np.ones(len(x)) / len(x) * 100,
        color=chart_colour,
        edgecolor="white"
    )
    plt.ylabel("Share (%)")
    plt.xlabel(col)
    plt.tight_layout()
    plt.savefig(chart_path / f"{col}_distribution_training.png")
    plt.show()
    

In [ ]:
# categorical feature distributions - training data only

for col in categorical_cols:
    counts = (
        training_df[col]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    counts.index = counts.index.astype(str)
    counts = counts * 100

    ax = counts.plot(kind="bar", color=chart_colour)

    ax.set_ylabel("Share (%)")
    ax.set_xlabel(col)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(chart_path / f"{col}_distribution_training.png")
    plt.show()

## 5. Summary tables of the data

In [ ]:
# summarise the training data, numeric and categorical variables

feature_cols = numeric_cols + categorical_cols

summary_wide = []

for col in feature_cols:
    column = training_df[col]

    row = {
        "variable": col,
        "type": "numeric" if col in numeric_cols else "categorical",
        "dtype": column.dtype,
        "n_unique": column.nunique(dropna=True),
        "missing_rate": column.isna().mean()
    }

    if col in numeric_cols:
        row.update({
            "min": column.min(),
            "median": column.median(),
            "mean": column.mean(),
            "max": column.max()
        })
    else:
        row.update({
            "min": np.nan,
            "median": np.nan,
            "mean": np.nan,
            "max": np.nan
        })

    summary_wide.append(row)

summary_wide_df = pd.DataFrame(summary_wide)

display(summary_wide_df)

In [ ]:
# make a more user friendly training data summary table

n_train = training_df.shape[0]
n_test = test_df.shape[0]

n_features = len(numeric_cols) + len(categorical_cols)
poverty_rate = training_df["poor"].mean()
avg_missing = training_df[numeric_cols + categorical_cols].isna().mean().mean()
max_unique = training_df[numeric_cols + categorical_cols].nunique().max()

training_df_overview = pd.DataFrame({
    "Data characteristic": [
        "Training observations",
        "Test observations",
        "Target",
        "Poverty rate in training data",
        "Raw features",
        "Numeric features",
        "Categorical features",
        "Average feature missingness",
        "Maximum unique values in any feature"
    ],
    "Value": [
        f"{n_train:,}",
        f"{n_test:,}",
        "Poor household indicator",
        f"{poverty_rate:.1%}",
        n_features,
        len(numeric_cols),
        len(categorical_cols),
        f"{avg_missing:.1%}",
        max_unique
    ]
})

display(training_df_overview)


In [ ]:
# numeric summary table

numeric_summary = training_df[numeric_cols].describe().T

numeric_summary["missing_rate"] = training_df[numeric_cols].isna().mean()
numeric_summary["n_unique"] = training_df[numeric_cols].nunique(dropna=True)

numeric_summary = (
    numeric_summary
    .reset_index()
    .rename(columns={"index": "variable", "50%": "median"})
)

numeric_summary = numeric_summary[
    ["variable", "missing_rate", "n_unique", "min", "median", "max"]
]

display(numeric_summary)

## 6. Export summary tables and data 

In [ ]:
# export tables and cleaned data

summary_wide_df.to_excel(table_path / "feature_summary_table_wide.xlsx", index=False)
training_df_overview.to_excel(table_path / "dataset_overview.xlsx", index=False)
numeric_summary.to_excel(table_path / "numeric_summary.xlsx", index=False)

training_df.to_csv(temp_path / "training_cleaned.csv", index=False)
test_df.to_csv(temp_path / "test_cleaned.csv", index=False)